In [1]:
import pyodbc 
import pandas as pd



In [2]:
#NOTES

##all new_events = to_insert_events

#new fights = fights_df

# new fight details  = new_fight_details

# fighters to update = fighters_df


# Θα κάνω σύνδεση με sql θα πάρω τα τελευταία events και θα πάω να κάνω screip αν υπάρχει νέο που δεν είναι σε αυτά τα 10 το πέρνω

In [3]:
import pyodbc

conn = pyodbc.connect(
    r"DRIVER={ODBC Driver 17 for SQL Server};"
    r"SERVER=MSI\SQLE22;"
    r"DATABASE=UFC;"
    r"Trusted_Connection=yes;"
)

print("CONNECTED")

CONNECTED


In [4]:
last_events_query ='select top 10 * from events order by id desc' 
last_events = pd.read_sql(last_events_query,conn)


C:\Users\mplan\AppData\Local\Temp\ipykernel_27548\2571800834.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  last_events = pd.read_sql(last_events_query,conn)


In [ ]:
import  requests
from bs4 import BeautifulSoup

print('ok')

URL = "http://ufcstats.com/statistics/events/completed?page=all"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(URL, headers=headers)
soup = BeautifulSoup(response.text, "html.parser")
rows = soup.select("tr.b-statistics__table-row")

events = []

for i,row in enumerate(rows):

    link = row.find("a")

    if link:

        event_name = link.text.strip()

        event_url = link["href"]

        cells = row.find_all("td")

        date = cells[0].text.strip()

        location = cells[1].text.strip()

        events.append({
            "event_name": event_name,
            "event_url": event_url,
            "date": date,
            "location": location
        })
    if i > 10 : 
        break

# θα πάω να πάρω τα τελευταία (πιο πρόσφατα 15)

ok


In [7]:
events

[]

In [8]:
# θα κάνω ένα καθάρισμα να τα φέρω σε σωστή μορφή 
#θα τα πάρω χωρίς id και αν δω κάποιο νέο θα πάω να το ρίξω με το νέο σωστό id 

df_events = pd.DataFrame(events)
df_events['date'] = df_events['date'].astype(str).str[-18:].str.strip()

month = []
day = []
year = []
for i in range(df_events.shape[0]):
    broken = df_events['date'][i].split(' ')
    month.append(broken[0])
    day.append(str(broken[1]).replace(',',''))
    year.append(broken[2])

df_events['day'] = day
df_events['month'] = month
df_events['year'] = year
df_events.drop(columns=['date'],inplace=True)

city=[]
region = []
country = []

for i in range(df_events.shape[0]):
    broken = df_events['location'][i].split(',')
    if len(broken) == 3:
        city.append(broken[0])
        region.append(broken[1])
        country.append(broken[2])


    if len(broken) <3 :
        city.append(broken[0])
        country.append(broken[1])
        region.append('unknown')
    
df_events['city'] = city
df_events['region'] = region
df_events['country'] = country

df_events.drop(columns=['location'],inplace=True)


card = []
for i in range(df_events.shape[0]):
    if 'UFC Fight Night' in df_events['event_name'][i]:
        card.append(0)
    else:
        card.append(1)
card

df_events['main_card'] = card
df_events = df_events[['event_name','main_card','year','country','month','day','city','region','event_url']]


KeyError: 'date'

In [ ]:
one = df_events['event_name']
zero = last_events['event_name']
new = set(one)-set(zero)

if len(new) == 0 :
    print('Δεν υπαρχουν νέα events')
else :
    to_insert_events = df_events[df_events['event_name'].isin(new)]

idd = int(last_events.iloc[0]['id'])
new_ids = [i+idd+1 for i in range(len(new))]

to_insert_events['ID'] = new_ids
to_insert_events = to_insert_events[['ID','event_name','main_card','year','country','month','day','city','region','event_url']]

In [ ]:
to_insert_events

,ID,event_name,main_card,year,country,month,day,city,region,event_url
0,774,UFC Fight Night: Allen vs. Costa,0,2026,USA,May,16,Las Vegas,Nevada,http://ufcstats.com/event-details/73abb7a5c57f...


# Είμαστε έτοιμοι με τον έλεγχο και ότι χρειάζεται για τα events τώρα πάμε στο επόμενο στάδιο που είναι ανα event να πάρουμε τα fight και τα fight_details 


In [ ]:
fight_links = to_insert_events['event_url'].tolist()

new_fights =[]

for link in fight_links:

    response = requests.get(link)

    soup = BeautifulSoup(response.text, "html.parser")

    rows = soup.select("tr.b-fight-details__table-row")

    event_name = soup.find(
        "span",
        class_="b-content__title-highlight"
    ).text.strip()

    for row in rows:

        fight_url = row.get("data-link")

        cells = row.find_all("td")

        if len(cells) < 10:
            continue

        fighters = cells[1].find_all("a")

        fighter_1 = fighters[0].text.strip() if len(fighters) > 0 else None
        fighter_2 = fighters[1].text.strip() if len(fighters) > 1 else None

        # stats
        kd = cells[2].text.strip()

        strikes = cells[3].text.strip()

        td = cells[4].text.strip()

        sub = cells[5].text.strip()

        # other info
        weight_class = cells[6].text.strip()

        method = cells[7].text.strip()

        round_ended = cells[8].text.strip()

        time = cells[9].text.strip()

        new_fights.append({

            "event_name": event_name,

            "fight_url": fight_url,

            "fighter_1": fighter_1,
            "fighter_2": fighter_2,

            "kd": kd,
            "strikes": strikes,
            "td": td,
            "sub": sub,

            "weight_class": weight_class,

            "method": method,

            "round": round_ended,

            "time": time
        })



In [ ]:
# πέιρα όλα τα fights που έγιναν στα events και τώρα πάω για καθάρισμα
# για τα kds 
fights_df = pd.DataFrame(new_fights)


fighter1_kd = []
fighter2_kd = []


kds = fights_df['kd']
for kd in kds:
    fighter1_kd.append(kd.replace('\n','')[:10].strip())
    fighter2_kd.append(kd.replace('\n','')[-10:].strip())


# για τα strikes 

fighter1_strikes = []
fighter2_strikes = []


strikes = fights_df['strikes']
for strike in strikes:
    fighter1_strikes.append(strike.replace('\n','')[:10].strip())
    fighter2_strikes.append(strike.replace('\n','')[-10:].strip())

# gia to td 


fighter1_td = []
fighter2_td = []


tds = fights_df['td']
for td in tds:
    fighter1_td.append(td.replace('\n','')[:10].strip())
    fighter2_td.append(td.replace('\n','')[-10:].strip())

# gia to sub 

fighter1_sub = []
fighter2_sub = []


subs = fights_df['sub']
for sub in subs:
    fighter1_sub.append(sub.replace('\n','')[:10].strip())
    fighter2_sub.append(sub.replace('\n','')[-10:].strip())


fights_df['fighter1_kd'] = fighter1_kd
fights_df['fighter2_kd'] = fighter2_kd

fights_df['fighter1_strikes'] = fighter1_strikes
fights_df['fighter2_strikes'] = fighter2_strikes

fights_df['fighter1_td'] = fighter1_td
fights_df['fighter2_td'] = fighter2_td

fights_df['fighter1_sub'] = fighter1_sub
fights_df['fighter2_sub'] = fighter2_sub


methods_d = fights_df['method']
methods = []
for method in methods_d:
    methods.append(method.replace('\n','').replace('              ',' ').strip())
fights_df['method'] = methods
fights_df.drop(columns=['kd','strikes','td','sub'],inplace=True)


# insert id 
id_for_fights = to_insert_events[['event_name','ID']]



fights_df = to_insert_events.merge(fights_df,on='event_name',how='left')

fights_df = fights_df.rename(columns={'ID': 'event_id'})




In [ ]:
fights_df.head(2)

,event_id,event_name,main_card,year,country,month,day,city,region,event_url,...,round,time,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub
0,774,UFC Fight Night: Allen vs. Costa,0,2026,USA,May,16,Las Vegas,Nevada,http://ufcstats.com/event-details/73abb7a5c57f...,...,5,5:00,1,0,98,100,7,0,0,0
1,774,UFC Fight Night: Allen vs. Costa,0,2026,USA,May,16,Las Vegas,Nevada,http://ufcstats.com/event-details/73abb7a5c57f...,...,2,4:29,1,0,72,72,0,0,0,0


In [ ]:
# τώρα θα πάρω τον νικιτή του κάθε αγώνα και θα κάνω ένωση 

import time

new_links = fights_df['fight_url'].tolist()

headers = {"User-Agent": "Mozilla/5.0"}

session = requests.Session()
session.headers.update(headers)

results = []

for i, url in enumerate(new_links):

    try:
        r = session.get(url, timeout=10)
        soup = BeautifulSoup(r.text, "html.parser")

        names = soup.select(".b-fight-details__person-name")
        statuses = soup.select(".b-fight-details__person-status")

        if len(names) < 2 or len(statuses) < 2:
            continue

        fighter_1 = names[0].get_text(strip=True)
        fighter_2 = names[1].get_text(strip=True)

        status_1 = statuses[0].get_text(strip=True)
        status_2 = statuses[1].get_text(strip=True)

        if status_1 == "W":
            winner = fighter_1
        elif status_2 == "W":
            winner = fighter_2
        else:
            winner = None

        results.append({
            "fight_url": url,
            "winner": winner
        })

        print(f"{i+1}/{len(new_links)}")

        time.sleep(0.5)  

    except Exception as e:
        print("error:", url)
        time.sleep(2)
        continue


wins = pd.DataFrame(results, columns=['fight_url', 'winner'])



1/13
2/13
3/13
4/13
5/13
6/13
7/13
8/13
9/13
10/13
11/13
12/13
13/13


In [ ]:
fights_df = fights_df.merge(wins, on='fight_url',how='left')
fights_df['winner'] = fights_df['winner'].fillna('draw')


In [ ]:
query_for_fight_ids = 'select top 1 fight_id from fights order by fight_id desc'


fight_ids = pd.read_sql(query_for_fight_ids,conn)
new_fight_ids = [i+int(fight_ids.loc[0].tolist()[0]) for i in range(len(fights_df['fight_url'].tolist()),0,-1)]
fights_df['fight_id'] = new_fight_ids


C:\Users\mplan\AppData\Local\Temp\ipykernel_25980\1356297371.py:4: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fight_ids = pd.read_sql(query_for_fight_ids,conn)


In [ ]:
import numpy as np

fights_df['fighter2_kd'] = fights_df['fighter2_kd'].replace('--', np.nan)
fights_df['fighter1_kd'] = fights_df['fighter1_kd'].replace('--', np.nan)
fights_df['fighter1_strikes'] = fights_df['fighter1_strikes'].replace('--', np.nan)
fights_df['fighter2_strikes'] = fights_df['fighter2_strikes'].replace('--', np.nan)
fights_df['fighter1_td'] = fights_df['fighter1_td'].replace('--', np.nan)
fights_df['fighter2_td'] = fights_df['fighter2_td'].replace('--', np.nan)
fights_df['fighter1_sub'] = fights_df['fighter1_sub'].replace('--', np.nan)
fights_df['fighter2_sub'] = fights_df['fighter2_sub'].replace('--', np.nan)

fights_df = fights_df[['fight_id','fight_url','fighter_1','fighter_2','winner','weight_class','method','round','time',
                       'event_name', 'event_id', 'fighter1_kd', 'fighter2_kd',
                       'fighter1_strikes', 'fighter2_strikes', 'fighter1_td', 'fighter2_td',
                       'fighter1_sub', 'fighter2_sub']]

# πολύ καλή δουλειά πάμε για fight_details


In [ ]:
fights_df.head(2)

,fight_id,fight_url,fighter_1,fighter_2,winner,weight_class,method,round,time,event_name,event_id,fighter1_kd,fighter2_kd,fighter1_strikes,fighter2_strikes,fighter1_td,fighter2_td,fighter1_sub,fighter2_sub
0,8701,http://ufcstats.com/fight-details/e4aa60812489...,Arnold Allen,Melquizael Costa,Arnold Allen,Featherweight,U-DEC,5,5:00,UFC Fight Night: Allen vs. Costa,774,1,0,98,100,7,0,0,0
1,8700,http://ufcstats.com/fight-details/fc1266e2892e...,Dooho Choi,Daniel Santos,Dooho Choi,Featherweight,KO/TKO Punch,2,4:29,UFC Fight Night: Allen vs. Costa,774,1,0,72,72,0,0,0,0


In [ ]:
new_fight_links = fights_df['fight_url'].tolist()

headers = {"User-Agent": "Mozilla/5.0"}




def split_two_values(text):
    parts = text.split()
    half = len(parts) // 2
    return " ".join(parts[:half]), " ".join(parts[half:])

def parse_fight(url):

    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # fighters
    fighters = [
        x.get_text(strip=True)
        for x in soup.select(".b-fight-details__person-name")
    ]

    if len(fighters) < 2:
        print(f"[SKIP] No fighters found: {url}")
        return None

    fighter1, fighter2 = fighters[0], fighters[1]

    # find sig strikes row
    rows = soup.select("tbody tr")

    sig_cols = None

    for row in rows:
        cols = [td.get_text(" ", strip=True) for td in row.find_all("td")]

        if len(cols) == 9 and "of" in cols[1] and "%" in cols[2]:
            sig_cols = cols
            break

    if not sig_cols:
        print(f"[SKIP] No stats row found: {url}")
        return None

    # split stats
    f1_sig_str, f2_sig_str = split_two_values(sig_cols[1])
    f1_sig_pct, f2_sig_pct = split_two_values(sig_cols[2])
    f1_head, f2_head = split_two_values(sig_cols[3])
    f1_body, f2_body = split_two_values(sig_cols[4])
    f1_leg, f2_leg = split_two_values(sig_cols[5])
    f1_distance, f2_distance = split_two_values(sig_cols[6])
    f1_clinch, f2_clinch = split_two_values(sig_cols[7])
    f1_ground, f2_ground = split_two_values(sig_cols[8])

    return {
        "url": url,
        "fighter1": fighter1,
        "fighter2": fighter2,

        "fighter1_sig_str": f1_sig_str,
        "fighter2_sig_str": f2_sig_str,

        "fighter1_sig_pct": f1_sig_pct,
        "fighter2_sig_pct": f2_sig_pct,

        "fighter1_head": f1_head,
        "fighter2_head": f2_head,

        "fighter1_body": f1_body,
        "fighter2_body": f2_body,

        "fighter1_leg": f1_leg,
        "fighter2_leg": f2_leg,

        "fighter1_distance": f1_distance,
        "fighter2_distance": f2_distance,

        "fighter1_clinch": f1_clinch,
        "fighter2_clinch": f2_clinch,

        "fighter1_ground": f1_ground,
        "fighter2_ground": f2_ground,
    }



results = []

for i, url in enumerate(new_fight_links, start=1):

    print(f"[{i}/{len(new_fight_links)}] Processing: {url}")

    try:
        data = parse_fight(url)

        if data:
            results.append(data)

    except Exception as e:
        print(f"[ERROR] {url} -> {e}")

    time.sleep(0.3)  


new_fight_details = pd.DataFrame(results)



[1/13] Processing: http://ufcstats.com/fight-details/e4aa608124896794
[2/13] Processing: http://ufcstats.com/fight-details/fc1266e2892ed111
[3/13] Processing: http://ufcstats.com/fight-details/ecb7ff543dd41bf8
[4/13] Processing: http://ufcstats.com/fight-details/57bd683efc797fc1
[5/13] Processing: http://ufcstats.com/fight-details/a6ec8573a9d38c51
[6/13] Processing: http://ufcstats.com/fight-details/ec32745308e2b055
[7/13] Processing: http://ufcstats.com/fight-details/6242cda790f2085d
[8/13] Processing: http://ufcstats.com/fight-details/c9e0aa88afb81dab
[9/13] Processing: http://ufcstats.com/fight-details/5f7111b982bcf6e3
[10/13] Processing: http://ufcstats.com/fight-details/48cb604345f11766
[11/13] Processing: http://ufcstats.com/fight-details/b98ead4eff6bf872
[12/13] Processing: http://ufcstats.com/fight-details/b482d4452e4d0eee
[13/13] Processing: http://ufcstats.com/fight-details/39417b7d07c2fd83


In [ ]:
# τώρα πρέπει να δώσουμε fight_id στα fight details 

fight_url = fights_df['fight_url']
fight_id = fights_df['fight_id']

fight_url_id = {
    fight_url[i]: int(fight_id[i])
    for i in range(len(fight_id))
}


fight_id_ = []

for i in range(len(fight_url)):
    fight_id_.append(fight_url_id.get(fight_url[i]))

new_fight_details['fight_id'] = fight_id_

# τώρα μέχρι εδώ έχουμε πάρει τα events όπως πρέπει , έχουμε πάρει τα fights και η μόνο εκρεμότητα που έχουν και αυτά αλλά και τα
# fight details είναι fighter ids όπου πρώτα πρέπει να πάρουμε τους fighters  να κάνουμε ανανέωση τα stats τους , να προσθέσουμε νέους αν υπάρχουν και 
# μετά να γίνει όλο το υπόλοιπο

### ο τρόπος να το κάνουμε αυτό θα είναι λίγο διαφορετικός , θα πέρνουμε τα ονόματα από όλους που είχαν αγώνα , θα ελέγχουμε στην βάση αν υπάρχει ήδη το όνομα ,
### αν υπάρχει θα κάνουμε αναζήτηση ξανά και θα ανανεώνουμε τα stats αλλιώς αν είναι νέος μαχητής θα ψάξουμε να βρούμε τα stats του 

In [7]:
get_all_fighter_names_query = 'select name from fighters'

names_af_all_fighters = pd.read_sql(get_all_fighter_names_query,conn)
names_f = names_af_all_fighters['name'].tolist()

C:\Users\mplan\AppData\Local\Temp\ipykernel_2296\3514268177.py:3: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  names_af_all_fighters = pd.read_sql(get_all_fighter_names_query,conn)


In [8]:
fighter1_names_from_new_fights = fights_df['fighter_1'].tolist()
fighter2_names_from_new_fights = fights_df['fighter_2'].tolist()

new_fighters = []
to_update_fighters = []

for i in range(len(fighter1_names_from_new_fights)):
    if fighter1_names_from_new_fights[i] not in names_f:
        new_fighters.append(fighter1_names_from_new_fights[i])
    else:
        to_update_fighters.append(fighter1_names_from_new_fights[i])

for i in range(len(fighter2_names_from_new_fights)):
    if fighter2_names_from_new_fights[i] not in names_f:
        new_fighters.append(fighter2_names_from_new_fights[i])
    else:
        to_update_fighters.append(fighter2_names_from_new_fights[i])

NameError: name 'fights_df' is not defined

In [9]:
# άρα προς το παρον έχουμε τα ονόματα και το τί πρέπει να κάνουμε 
#ξεκινάμε με το update που είναι πιο έυκολο

In [10]:
update_ = ','.join(['?'] * len(to_update_fighters))

get_urls_from_players_to_update = f"""
SELECT url
FROM fighters
WHERE name IN ({update_})
"""

updated_urls_from_fighters = pd.read_sql(
    get_urls_from_players_to_update,
    conn,
    params=to_update_fighters
)

NameError: name 'to_update_fighters' is not defined

In [11]:
#έχουμε τα λινκς τώρα θα θεωρήσουμε ότι εφόσον έγιναν τα μάτς θα έχουν ανανεωθεί άρα θα πάμε να τα ξανακάνουμε σκρειπ

In [12]:

headers = {
    "User-Agent": "Mozilla/5.0"
}


fighter_links = updated_urls_from_fighters['url'].tolist()
results = []

for i, url in enumerate(fighter_links):

    try:

        r = requests.get(url, headers=headers, timeout=10)

        soup = BeautifulSoup(r.text, "html.parser")



        name_tag = soup.select_one(".b-content__title-highlight")

        name = name_tag.text.strip() if name_tag else None

        nickname_tag = soup.select_one(".b-content__Nickname")

        nickname = nickname_tag.text.strip() if nickname_tag else None

        record_tag = soup.select_one(".b-content__title-record")

        record_text = record_tag.text.strip() if record_tag else ""

        record = record_text.replace("Record:", "").strip()

        wins = None
        losses = None
        draws = None

        try:
            wins, losses, draws = record.split("-")
        except:
            pass

        info = {}

        rows = soup.select(".b-list__box-list-item")

        for row in rows:

            text = row.text.strip().replace("\n", "")

            if ":" in text:

                key, value = text.split(":", 1)

                info[key.strip()] = value.strip()


        fighter_data = {
            "url": url,
            "name": name,
            "nickname": nickname,
            "record": record,
            "wins": wins,
            "losses": losses,
            "draws": draws,
        }

        # προσθέτει όλα τα stats/info
        fighter_data.update(info)

        # append row
        results.append(fighter_data)

        print(f"{i+1}/{len(fighter_links)} DONE -> {name}")

        time.sleep(0.5)

    except Exception as e:

        print(f"\nERROR AT {url}")
        print(e)


fighters_df = pd.DataFrame(results)


fighters_df[['wins', 'losses', 'draws']] = (
    fighters_df[['wins', 'losses', 'draws']]
    .apply(pd.to_numeric, errors='coerce')
)



NameError: name 'updated_urls_from_fighters' is not defined

In [13]:
#έχω μείνει εκεί που πρέπει να κάνω pdate τα στατς των μαχητών 